# CivicStruct supplemental constrained-generation run

Run every cell from the top in a fresh Kaggle GPU session. This is an inference-only, post-hoc supplemental experiment. It does not load the QLoRA adapter or change Evaluation v2.

In [ ]:
%pip -q install --disable-pip-version-check 'transformers==5.10.1' bitsandbytes accelerate 'lm-format-enforcer==0.10.11' 'mlflow==3.15.1'

from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

REPO_REF = '6c1c733'
RAW_BASE = f'https://raw.githubusercontent.com/goyashek/civic-grievance-structurer/{REPO_REF}'
ROOT = Path('/kaggle/working/civicstruct')
ROOT.mkdir(parents=True, exist_ok=True)
FETCH_FILES = (
    'src/evaluate.py',
    'src/schema.py',
    'data/dataset_manifest.json',
    'data/surface_variants.jsonl',
    'data/test_cases.jsonl',
    'data/external_civic_eval.jsonl',
)

def fetch(relative_path):
    destination = ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    try:
        with urlopen(f'{RAW_BASE}/{relative_path}', timeout=60) as response:
            destination.write_bytes(response.read())
    except URLError as exc:
        raise RuntimeError('Turn on Kaggle Internet to fetch the frozen project files.') from exc

for relative_path in FETCH_FILES:
    fetch(relative_path)
print({'source': RAW_BASE, 'fetched': list(FETCH_FILES), 'training_loaded': False})

In [ ]:
import hashlib
import json
import os
import platform
import shutil
import sys
import time
from importlib.metadata import version

import mlflow
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, PreTrainedTokenizerBase
import transformers.tokenization_utils as tokenization_utils

# lm-format-enforcer 0.10.11 uses the old Transformers module path.
# Transformers 5 still exports the class at its package root.
tokenization_utils.PreTrainedTokenizerBase = PreTrainedTokenizerBase
from lmformatenforcer import JsonSchemaParser
from lmformatenforcer.integrations.transformers import build_transformers_prefix_allowed_tokens_fn

sys.path.insert(0, str(ROOT))
from src.evaluate import evaluate_outputs

SEED = 42
MODEL_NAME = 'HuggingFaceTB/SmolLM3-3B'
MODEL_REVISION = 'a07cc9a04f16550a088caea529712d1d335b0ac1'
DATASET_VERSION = 'frozen_full_v2'
PROMPT_VERSION = 'posthoc_constrained_json_schema_v1'
DECODER_NAME = 'lm-format-enforcer'
MAX_NEW_TOKENS = 256
MODEL_DTYPE = torch.float16
OUTPUT = Path('/kaggle/working/civicstruct_constrained_output')
OUTPUT.mkdir(parents=True, exist_ok=True)
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
mlflow.set_tracking_uri((OUTPUT / 'mlruns').as_uri())
mlflow.set_experiment('civicstruct-supplemental')
torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Select a Kaggle GPU runtime before running this notebook.'
torch.cuda.manual_seed_all(SEED)

def json_default(value):
    if isinstance(value, set):
        return sorted(value, key=str)
    if isinstance(value, (Path, os.PathLike)):
        return os.fspath(value)
    if isinstance(value, torch.dtype):
        return str(value)
    if hasattr(value, 'item'):
        try:
            return value.item()
        except (TypeError, ValueError):
            pass
    if hasattr(value, 'tolist'):
        return value.tolist()
    return str(value)

def save_json(path, value):
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=json_default) + '\n', encoding='utf-8')

def load_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

manifest = json.loads((ROOT / 'data/dataset_manifest.json').read_text(encoding='utf-8'))
for relative_path in ('data/surface_variants.jsonl', 'data/test_cases.jsonl', 'data/external_civic_eval.jsonl'):
    assert sha256(ROOT / relative_path) == manifest['sha256'][relative_path], relative_path

surfaces = load_jsonl(ROOT / 'data/surface_variants.jsonl')
validation = [row for row in surfaces if row['split'] == 'validation']
internal_test = load_jsonl(ROOT / 'data/test_cases.jsonl')
external_transfer = load_jsonl(ROOT / 'data/external_civic_eval.jsonl')
for row in validation + internal_test + external_transfer:
    row.setdefault('surface_id', row['case_id'])

assert len(validation) == 50
assert len(internal_test) == 50
assert len(external_transfer) == 20
assert len({row['surface_id'] for row in validation}) == len(validation)
assert len({row['case_id'] for row in internal_test}) == len(internal_test)
assert len({row['case_id'] for row in external_transfer}) == len(external_transfer)
SPLITS = {
    'validation': validation,
    'internal_test_posthoc': internal_test,
    'external_transfer_posthoc': external_transfer,
}
print({'device': torch.cuda.get_device_name(0), 'python': platform.python_version(), 'torch': torch.__version__})
print({name: len(rows) for name, rows in SPLITS.items()}, {'dataset_version': manifest['dataset_version'], 'training_loaded': False})

In [ ]:
DOMAINS = ['public_transport', 'water_supply', 'sanitation_and_waste', 'roads_and_streetlights', 'electricity', 'welfare_or_document_service', 'other']
ISSUES = ['delay_or_non_arrival', 'service_outage_or_non_delivery', 'damaged_infrastructure', 'overcharging_or_payment_problem', 'record_or_document_error', 'staff_conduct', 'safety_or_health_hazard', 'other']
URGENCY = ['routine', 'time_sensitive', 'safety_critical']
MISSING = ['exact_location', 'date_or_time', 'service_identifier', 'transaction_or_reference_id', 'amount', 'supporting_evidence', 'affected_person_or_group', 'none']
STRUCTURED_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'required': ['service_domain', 'issue_type', 'location', 'event_date_or_time', 'amount_inr', 'service_identifier', 'urgency', 'missing_information', 'formal_summary'],
    'properties': {
        'service_domain': {'type': 'string', 'enum': DOMAINS},
        'issue_type': {'type': 'string', 'enum': ISSUES},
        'location': {'type': ['string', 'null']},
        'event_date_or_time': {'type': ['string', 'null']},
        'amount_inr': {'type': ['number', 'null'], 'minimum': 0},
        'service_identifier': {'type': ['string', 'null']},
        'urgency': {'type': 'string', 'enum': URGENCY},
        'missing_information': {'type': 'array', 'items': {'type': 'string', 'enum': MISSING}, 'minItems': 1, 'maxItems': len(MISSING)},
        'formal_summary': {'type': 'string', 'minLength': 1},
    },
}
SYSTEM_PROMPT = (
    'Structure one public-service complaint as exactly one JSON object. '
    'Use the nine required fields from the task schema, null for absent scalar facts, and no invented facts. '
    'Keep missing_information in label order and use none only when it is the only item. '
    'Return no reasoning, markdown, or commentary. The decoder enforces the JSON Schema.'
)

def messages_for(complaint):
    return [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': complaint}]

parser = JsonSchemaParser(STRUCTURED_SCHEMA)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
tokenizer.padding_side = 'left'
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=MODEL_DTYPE, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, quantization_config=quantization_config, dtype=MODEL_DTYPE, device_map='auto')
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.eos_token_id
model.config.use_cache = True
model.eval()
prefix_allowed_tokens_fn = build_transformers_prefix_allowed_tokens_fn(tokenizer, parser)
print({'model': MODEL_NAME, 'revision': MODEL_REVISION, 'decoder': DECODER_NAME, 'missing_information_labels': len(MISSING)})

In [ ]:
def encoded_messages(messages):
    kwargs = {'tokenize': True, 'add_generation_prompt': True, 'return_dict': True, 'return_tensors': 'pt', 'padding': True, 'truncation': True, 'max_length': 2048, 'enable_thinking': False}
    try:
        return tokenizer.apply_chat_template(messages, **kwargs)
    except TypeError:
        kwargs.pop('enable_thinking')
        return tokenizer.apply_chat_template(messages, **kwargs)

def generate_rows(rows):
    outputs = []
    input_device = next(model.parameters()).device
    for row in rows:
        inputs = encoded_messages([messages_for(row['complaint'])]).to(input_device)
        prompt_tokens = int(inputs['attention_mask'].sum().item())
        padded_length = inputs['input_ids'].shape[1]
        torch.cuda.synchronize()
        started = time.perf_counter()
        with torch.inference_mode():
            generated = model.generate(**inputs, do_sample=False, max_new_tokens=MAX_NEW_TOKENS, use_cache=True, pad_token_id=tokenizer.pad_token_id, prefix_allowed_tokens_fn=prefix_allowed_tokens_fn)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - started
        outputs.append({'surface_id': row['surface_id'], 'case_id': row['case_id'], 'response': tokenizer.decode(generated[0, padded_length:], skip_special_tokens=True).strip(), 'latency_seconds': elapsed, 'prompt_tokens': prompt_tokens})
    return outputs

In [ ]:
def score_split(name, rows, outputs):
    scores = evaluate_outputs([row['gold'] for row in rows], [item['response'] for item in outputs])
    record = {'split': name, 'experiment_type': 'supplemental_posthoc', 'model_name': MODEL_NAME, 'model_revision': MODEL_REVISION, 'dataset_version': DATASET_VERSION, 'prompt_version': PROMPT_VERSION, 'decoder': DECODER_NAME, 'decoding': {'do_sample': False, 'max_new_tokens': MAX_NEW_TOKENS}, 'mean_latency_seconds': sum(item['latency_seconds'] for item in outputs) / len(outputs), 'mean_prompt_tokens': sum(item['prompt_tokens'] for item in outputs) / len(outputs), 'scores': scores, 'outputs': outputs}
    save_json(OUTPUT / f'{name}_results.json', record)
    strict = scores['strict']
    fields = strict['end_to_end_field_metrics']
    print({'split': name, 'schema_validity': strict['schema_validity_rate'], 'domain_f1': fields['service_domain_macro_f1'], 'issue_f1': fields['issue_type_macro_f1'], 'missing_information_f1': fields['missing_information_macro_f1'], 'fact_mismatch_rate': strict['exact_factual_field_mismatches']['rate']})
    return record

results = {}
for split_name, rows in SPLITS.items():
    results[split_name] = score_split(split_name, rows, generate_rows(rows))

In [ ]:
packages = {name: version(name) for name in ('torch', 'transformers', 'bitsandbytes', 'accelerate', 'lm-format-enforcer', 'mlflow')}
summary = {'experiment_type': 'supplemental_posthoc', 'test_boundary': 'internal test and external transfer are post-hoc comparisons; Evaluation v2 is unchanged', 'model_name': MODEL_NAME, 'model_revision': MODEL_REVISION, 'dataset_version': DATASET_VERSION, 'prompt_version': PROMPT_VERSION, 'decoder': DECODER_NAME, 'decoder_schema': STRUCTURED_SCHEMA, 'packages': packages, 'device': torch.cuda.get_device_name(0), 'results': results}
with mlflow.start_run(run_name='smollm3_constrained_json_posthoc') as run:
    summary['mlflow_run_id'] = run.info.run_id
    mlflow.log_params({'method': 'base_model_constrained_json_schema', 'model_name': MODEL_NAME, 'model_revision': MODEL_REVISION, 'dataset_version': DATASET_VERSION, 'prompt_version': PROMPT_VERSION, 'decoder': DECODER_NAME, 'decoder_version': packages['lm-format-enforcer'], 'evaluator_version': '2.0', 'metric_impact': 'supplemental_only'})
    for split_name, record in results.items():
        strict = record['scores']['strict']
        fields = strict['end_to_end_field_metrics']
        metrics = {f'{split_name}_schema_validity': strict['schema_validity_rate'], f'{split_name}_domain_f1': fields['service_domain_macro_f1'], f'{split_name}_issue_f1': fields['issue_type_macro_f1'], f'{split_name}_missing_information_f1': fields['missing_information_macro_f1'], f'{split_name}_mean_latency_seconds': record['mean_latency_seconds']}
        fact_rate = strict['exact_factual_field_mismatches']['rate']
        if fact_rate is not None:
            metrics[f'{split_name}_fact_mismatch_rate'] = fact_rate
        mlflow.log_metrics(metrics)
    save_json(OUTPUT / 'supplemental_results.json', summary)
    mlflow.log_artifact(str(OUTPUT / 'supplemental_results.json'))
    for path in OUTPUT.glob('*_results.json'):
        mlflow.log_artifact(str(path))
save_json(OUTPUT / 'supplemental_results.json', summary)
zip_path = shutil.make_archive(str(OUTPUT), 'zip', root_dir=OUTPUT)
print({'mlflow_run_id': summary['mlflow_run_id'], 'results_zip': zip_path, 'output_dir': str(OUTPUT)})

The output ZIP contains raw responses, Evaluation v2 scores, the decoder schema, package versions, and the MLflow store. Keep these results outside the original frozen model-selection table.